# part 1

In [1]:
# !pip install openpyxl

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

In [3]:
fp_avail="gs://agntworks-data-dev/sandbox/experiments/Available Flights Last 18 mos.xlsx"
fp_comp="gs://agntworks-data-dev/sandbox/experiments/Completed-Flights-2016.xlsx"
fp_quotes="gs://agntworks-data-dev/sandbox/experiments/XODM-quotes-trips-2016.xlsx"

avail = pd.read_excel(fp_avail)
completed = pd.read_excel(fp_comp)
quotes_2016 = pd.read_excel(fp_quotes, sheet_name='2016')

In [4]:
def _to_timedelta(s: pd.Series) -> pd.Series:
    # Handles strings like '02:44:00' and missing values
    return pd.to_timedelta(s.astype('string'), errors='coerce')


# Trip-level aggregation from completed legs
completed_leg = completed.copy()
completed_leg['Dep_DT_GMT'] = pd.to_datetime(completed_leg['Dep_Date_Actual_GMT']) + _to_timedelta(completed_leg['Dep_Time_Actual_GMT'])
completed_leg['Arr_DT_GMT'] = pd.to_datetime(completed_leg['Arr_Date_Actual_GMT']) + _to_timedelta(completed_leg['Arr_Time_Actual_GMT'])

block_min = (completed_leg['Arr_DT_GMT'] - completed_leg['Dep_DT_GMT']).dt.total_seconds() / 60
completed_leg['Block_Min'] = block_min.where(block_min > 0)

for c in ['Statute_Miles', 'PAX_Count', 'DHflag', 'Leg_Number', 'Leg_Count']:
    completed_leg[c] = pd.to_numeric(completed_leg[c], errors='coerce')

trip_agg = (
    completed_leg
    .groupby('Trip_Number', dropna=False)
    .agg(
        n_legs=('Trip_Legs_ID', 'nunique'),
        leg_count_max=('Leg_Count', 'max'),
        miles_total=('Statute_Miles', 'sum'),
        block_min_total=('Block_Min', 'sum'),
        pax_max=('PAX_Count', 'max'),
        dh_legs=('DHflag', 'sum'),
        miles_dh=('Statute_Miles', lambda x: x[completed_leg.loc[x.index, 'DHflag'].fillna(0).astype(int) == 1].sum()),
        dep_dt_min=('Dep_DT_GMT', 'min'),
        arr_dt_max=('Arr_DT_GMT', 'max'),
        dep_icao_first=('Dep_ICAO', 'first'),
        arr_icao_last=('Arr_ICAO', 'last'),
        trip_type_code=('Trip_type_code', 'first'),
        trip_purpose=('Trip_Purpose', 'first'),
        aircraft_type_id=('Aircraft_Type_ID', 'first'),
        aircraft_id=('Aircraft_ID', 'first'),
        quote_total_cost=('Quote_Total_Cost', 'first'),
        ac_cost_total=('AC_Cost_Total', 'first'),
    )
    .reset_index()
)

trip_agg['dh_share_miles'] = (trip_agg['miles_dh'] / trip_agg['miles_total']).replace([np.inf, -np.inf], np.nan)
trip_agg['block_hr_total'] = trip_agg['block_min_total'] / 60
trip_agg['speed_mph'] = (trip_agg['miles_total'] / trip_agg['block_hr_total']).replace([np.inf, -np.inf], np.nan)
trip_agg['cost_per_mile_actual'] = (trip_agg['quote_total_cost'] / trip_agg['miles_total']).replace([np.inf, -np.inf], np.nan)
trip_agg['cost_per_block_hr_actual'] = (trip_agg['quote_total_cost'] / trip_agg['block_hr_total']).replace([np.inf, -np.inf], np.nan)
trip_agg['margin_vs_ac_cost'] = trip_agg['quote_total_cost'] - trip_agg['ac_cost_total']
trip_agg['margin_pct_vs_ac_cost'] = (trip_agg['margin_vs_ac_cost'] / trip_agg['quote_total_cost']).replace([np.inf, -np.inf], np.nan)

trip_agg['dep_month'] = trip_agg['dep_dt_min'].dt.to_period('M').astype('string')
trip_agg['dep_dow'] = trip_agg['dep_dt_min'].dt.day_name()

# Join latest quote row per Trip_Number
quotes_trip = quotes_2016.copy()
quotes_trip['Trip_Number'] = pd.to_numeric(quotes_trip['Trip_Number'], errors='coerce')
quotes_trip = quotes_trip.sort_values('Last_Update_Date').drop_duplicates(subset=['Trip_Number'], keep='last')

trip = trip_agg.merge(
    quotes_trip[[
        'Trip_Number', 'Trip_Status', 'Date_Quoted', 'Scheduled_Date', 'Trip_Departure_Date', 'Trip_Arrival_Date',
        'Percent_Discount', 'Quote_Status', 'Quoted_Route', 'Live_Leg_City_Pairs', 'qTrip_type_code', 'qRate_Type_Code',
        'Aircraft_Rate', 'DHRate', 'Company_Code', 'Quoted_For_Company', 'Last_Update_Date'
    ]],
    on='Trip_Number',
    how='left'
)

# Basic cleaning consistent with earlier notebook
trip_cln = trip.copy()
trip_cln.loc[trip_cln['miles_total'] <= 0, ['cost_per_mile_actual']] = np.nan
trip_cln.loc[trip_cln['block_hr_total'] <= 0, ['cost_per_block_hr_actual', 'speed_mph']] = np.nan
trip_cln = trip_cln[trip_cln['quote_total_cost'] >= 0].copy()

trip_realized = trip_cln[trip_cln['Quote_Status'].fillna('NA').isin(['Invoiced - Ready to Export', 'Paid - Exported to Accounting'])].copy()

In [5]:
# Booking lead time features
trip_realized['Date_Quoted'] = pd.to_datetime(trip_realized['Date_Quoted'], errors='coerce')
trip_realized['Trip_Departure_Date'] = pd.to_datetime(trip_realized['Trip_Departure_Date'], errors='coerce')
trip_realized['lead_time_days'] = (trip_realized['Trip_Departure_Date'] - trip_realized['Date_Quoted']).dt.total_seconds() / (3600 * 24)
trip_realized.loc[trip_realized['lead_time_days'] < 0, 'lead_time_days'] = np.nan  # guard against dirty timestamps

trip_realized['quoted_dow'] = trip_realized['Date_Quoted'].dt.day_name()
trip_realized['quoted_month'] = trip_realized['Date_Quoted'].dt.to_period('M').astype('string')

print('trip_realized shape:', trip_realized.shape)
print('Customers (Quoted_For_Company) non-null:', trip_realized['Quoted_For_Company'].notna().sum())
print('Lead time available (non-null):', trip_realized['lead_time_days'].notna().sum())
trip_realized[['Trip_Number','Quoted_For_Company','Date_Quoted','Trip_Departure_Date','lead_time_days','quoted_dow','dep_dow','quote_total_cost']].head(5)

trip_realized shape: (3835, 46)
Customers (Quoted_For_Company) non-null: 3835
Lead time available (non-null): 3831


,Trip_Number,Quoted_For_Company,Date_Quoted,Trip_Departure_Date,lead_time_days,quoted_dow,dep_dow,quote_total_cost
3,162484,MacGregor & Mary Read Revocable Trust - PRF6,2016-01-14,2016-04-07,84.0,Thursday,Friday,158855.00
54,169174,Arthur Hershaft - PRF6,2016-01-04,2016-01-25,21.0,Monday,Monday,26800.00
68,169809,Stone Ridge Asset Management - PRF6,2016-03-28,2016-03-21,NaN,Monday,Monday,88591.31
84,170330,Patricia Hearst Shaw - PRF2,2016-01-01,2016-01-01,0.0,Friday,Friday,34561.76
409,174203,Mark Burnett Productions - PRF5,2016-01-01,2016-01-05,4.0,Friday,Tuesday,26225.08
